# Qwen3.5-0.8B LoRA Fine-Tuning — v3 (Colab)

**Project:** Local SLM / Forecasting-REI  
**Model:** `Qwen/Qwen3.5-0.8B`  
**Method:** LoRA supervised fine-tuning (SFT), completion-only loss  
**Corpus:** `forecasting-llmrei-extraction-1049-v3` (865 train / 184 validation)  
**Repo:** https://github.com/AbdullahUsman0/SLM-FineTuning-Testing

---

## Before running

1. Go to **Runtime > Change runtime type > T4 GPU**
2. Connect Google Drive when prompted in Cell 1
3. Run cells **in order** — each cell depends on the one above

**Why v3 failed before:** Colab had `torchao 0.10.0`; PEFT requires `>=0.16.0`.  
Cell 2 fixes this before any training code runs.

## Cell 1 — Mount Drive & Clone Repos

In [ ]:
import subprocess
import sys
from pathlib import Path
from google.colab import drive

PROJECT_DIR  = Path('/content/local-slm-lab')
FPY_DIR      = Path('/content/fpy')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v3')

# Mount Drive
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
else:
    print('Drive already mounted.')

def run(cmd, cwd=None):
    print('>', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

# Clone / update SLM repo
if not (PROJECT_DIR / '.git').is_dir():
    run(['git', 'clone', 'https://github.com/AbdullahUsman0/SLM-FineTuning-Testing.git', PROJECT_DIR])
else:
    print('Repo already cloned - pulling latest main...')
    run(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', 'main'])

# Clone / update fpy at pinned commit
FPY_COMMIT = '04d52c015d1e3ecdefe92b87116f209361509b4b'
if not (FPY_DIR / '.git').is_dir():
    run(['git', 'clone', 'https://github.com/int-abd-5/fpy.git', FPY_DIR])
run(['git', '-C', FPY_DIR, 'fetch', '--depth=1', 'origin', FPY_COMMIT])
run(['git', '-C', FPY_DIR, 'checkout', '--detach', FPY_COMMIT])

run(['git', '-C', PROJECT_DIR, 'log', '--oneline', '-5'])
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f'Drive output directory: {DRIVE_OUTPUT}')

## Cell 2 — Fix torchao & Install Requirements

**Root cause of the v3 failure:** `torchao 0.10.0` was preinstalled;  
PEFT requires `>=0.16.0`. This cell upgrades it FIRST, then installs the  
pinned training requirements.

In [ ]:
import importlib.metadata
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/local-slm-lab')
FPY_DIR     = Path('/content/fpy')

# 1. Upgrade torchao first (fixes PEFT LoRA injection error)
print('[1/4] Upgrading torchao to >=0.16.0 ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'torchao>=0.16.0'],
    check=True
)
print('torchao version:', importlib.metadata.version('torchao'))

# 2. Install pinned training requirements
print('\n[2/4] Installing training/requirements.txt ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r',
     str(PROJECT_DIR / 'training' / 'requirements.txt')],
    check=True
)

# 3. Install fpy (read-only forecasting schema)
print('\n[3/4] Installing fpy (editable) ...')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-e', str(FPY_DIR)],
    check=True
)

# 4. Print final environment
print('\n[4/4] Environment summary:')
packages = ['torch', 'torchao', 'transformers', 'datasets',
             'accelerate', 'peft', 'trl', 'safetensors']
for pkg in packages:
    try:
        print(f'  {pkg}: {importlib.metadata.version(pkg)}')
    except importlib.metadata.PackageNotFoundError:
        print(f'  {pkg}: MISSING')

import torch
print(f'\nCUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 3 — Build & Verify Corpus v3 + Run Tests

**Fix applied here:** The tests import `from local_slm_lab...` and `from training.train_lora...`  
which both require the project root on `PYTHONPATH`. We set that explicitly and also  
ensure `training/` is importable as a package before running the test suite.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path('/content/local-slm-lab')

# ── Build corpus ───────────────────────────────────────────────────────────────
print('Building corpus-v3 ...')
subprocess.run(
    [sys.executable, 'scripts/build-corpus-v3.py'],
    cwd=PROJECT_DIR, check=True
)

# ── Verify checksums ───────────────────────────────────────────────────────────
print('\nVerifying corpus-v3 checksums ...')
subprocess.run(
    [sys.executable, 'scripts/verify-corpus-v3.py'],
    cwd=PROJECT_DIR, check=True
)

# ── Ensure training/ is importable as a package ────────────────────────────────
# training/ has no __init__.py so 'from training.train_lora import ...' fails
# unless we create one (or add PROJECT_DIR to sys.path, which discover does).
init_file = PROJECT_DIR / 'training' / '__init__.py'
if not init_file.exists():
    init_file.write_text('', encoding='utf-8')
    print('Created training/__init__.py (needed for test import)')

# ── Build env with project root on PYTHONPATH ──────────────────────────────────
env = os.environ.copy()
existing_pypath = env.get('PYTHONPATH', '')
env['PYTHONPATH'] = str(PROJECT_DIR) + ((':' + existing_pypath) if existing_pypath else '')

# ── Run tests using discover (avoids direct module-name resolution issues) ─────
# Run the two target test files individually via discover --pattern
for test_file in ['test_corpus_v3.py', 'test_training_helpers.py']:
    print(f'\nRunning tests/{test_file} ...')
    result = subprocess.run(
        [sys.executable, '-m', 'unittest', 'discover',
         '--start-directory', 'tests',
         '--pattern', test_file,
         '--verbose'],
        cwd=PROJECT_DIR,
        env=env
    )
    if result.returncode != 0:
        raise RuntimeError(f'Tests failed for {test_file} (exit code {result.returncode})')

# ── Record counts ──────────────────────────────────────────────────────────────
train_path = PROJECT_DIR / 'corpus-v3' / 'sft' / 'train.jsonl'
val_path   = PROJECT_DIR / 'corpus-v3' / 'sft' / 'validation.jsonl'
train_count = sum(1 for _ in train_path.open('r', encoding='utf-8'))
val_count   = sum(1 for _ in val_path.open('r', encoding='utf-8'))
print(f'\nCorpus v3 ready:  {train_count} train  |  {val_count} validation')
assert train_count == 865, f'Expected 865 train records, got {train_count}'
assert val_count   == 184, f'Expected 184 validation records, got {val_count}'
print('OK - All checks passed')

## Cell 4 — Train LoRA v3

| Hyperparameter | Value |
|---|---|
| Base model | `Qwen/Qwen3.5-0.8B` |
| Epochs | 3 (+ early stopping patience 1) |
| Learning rate | 5e-5 |
| LoRA rank / alpha | 16 / 32 |
| Effective batch | 16 |
| Max length | 3,072 |
| Loss scope | completion-only |
| Checkpoint resume | auto (safe to re-run if Colab disconnects) |

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_DIR  = Path('/content/local-slm-lab')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v3')
DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)

command = [
    sys.executable, 'training/train_lora.py',
    '--train',      'corpus-v3/sft/train.jsonl',
    '--validation', 'corpus-v3/sft/validation.jsonl',
    '--output',     str(DRIVE_OUTPUT),
    '--epochs',     '3',
    '--learning-rate', '5e-5',
    '--early-stopping-patience', '1',
    '--max-length', '3072',
    '--resume-from-checkpoint', 'auto',
]

print('Starting v3 training (or resuming from highest checkpoint if Colab disconnected)')
print('Command:', ' '.join(command))
print('-' * 60)

result = subprocess.run(command, cwd=PROJECT_DIR)

print('-' * 60)
if result.returncode == 0:
    print('Training completed successfully')
else:
    print(f'Training exited with code {result.returncode}')
    raise RuntimeError('Training failed - check output above for details')

## Cell 5 — Verify Saved Adapter & Compute SHA-256

In [ ]:
import hashlib
import json
from pathlib import Path

DRIVE_OUTPUT = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v3')

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest().upper()

adapter_file  = DRIVE_OUTPUT / 'best-adapter' / 'adapter_model.safetensors'
eval_file     = DRIVE_OUTPUT / 'evaluation.json'
manifest_file = DRIVE_OUTPUT / 'run-manifest.json'

print('=' * 60)
print('POST-TRAINING VERIFICATION')
print('=' * 60)

if adapter_file.is_file():
    checksum = sha256(adapter_file)
    print(f'OK  best-adapter/adapter_model.safetensors')
    print(f'    Size:   {adapter_file.stat().st_size:,} bytes')
    print(f'    SHA256: {checksum}')
else:
    print('MISSING  adapter_model.safetensors')

checkpoints = sorted(
    [p.name for p in DRIVE_OUTPUT.glob('checkpoint-*') if p.is_dir()]
)
print(f'\nCheckpoints saved: {checkpoints}')

if eval_file.is_file():
    metrics = json.loads(eval_file.read_text(encoding='utf-8'))
    print('\nFinal evaluation metrics:')
    for k, v in metrics.items():
        print(f'  {k}: {v}')
else:
    print('MISSING  evaluation.json')

if manifest_file.is_file():
    manifest = json.loads(manifest_file.read_text(encoding='utf-8'))
    print('\nEnvironment recorded in run-manifest.json:')
    for pkg, ver in manifest.get('packages', {}).items():
        print(f'  {pkg}: {ver}')
    print(f"  GPU: {manifest.get('gpu', 'unknown')}")
    print(f"  Python: {manifest.get('python', 'unknown')}")

## Cell 6 — Extraction Smoke Test

Loads the best adapter and runs a single extraction call to confirm  
the model responds with valid JSON and is not producing empty updates.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_DIR  = Path('/content/local-slm-lab')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v3')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

import torch
from peft import PeftModel
from transformers import AutoProcessor, AutoModelForCausalLM

BASE_MODEL   = 'Qwen/Qwen3.5-0.8B'
ADAPTER_PATH = str(DRIVE_OUTPUT / 'best-adapter')

print(f'Loading base model: {BASE_MODEL}')
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
processor  = AutoProcessor.from_pretrained(BASE_MODEL)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=dtype, device_map='auto'
)
print(f'Loading LoRA adapter from: {ADAPTER_PATH}')
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()
print('Model ready.')

SMOKE_PROMPT = [
    {
        'role': 'system',
        'content': (
            'You are an extraction assistant. '
            'Extract slot values mentioned in the user message. '
            'Return only a JSON object with slot_updates (dict) and intent (string).'
        )
    },
    {
        'role': 'user',
        'content': 'I want to forecast monthly electricity demand in Pakistan for the next 12 months.'
    }
]

inputs = processor.apply_chat_template(
    SMOKE_PROMPT, tokenize=True, add_generation_prompt=True, return_tensors='pt'
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        inputs, max_new_tokens=256, do_sample=False, temperature=None, top_p=None
    )

new_tokens = output_ids[0][inputs.shape[-1]:]
response = processor.decode(new_tokens, skip_special_tokens=True)
print('\n=== Model response ===')
print(response)

try:
    parsed  = json.loads(response)
    updates = parsed.get('slot_updates', {})
    intent  = parsed.get('intent', '')
    print(f'\nOK  Valid JSON - intent={intent!r}, slot_updates={updates}')
    if updates:
        print('OK  Non-empty slot updates (no empty-update collapse)')
    else:
        print('WARN  slot_updates is empty - may indicate empty-update collapse')
except json.JSONDecodeError as e:
    print(f'FAIL  JSON parse error: {e}')

## Cell 7 — Save Run Summary to Drive

In [ ]:
import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path

import torch

DRIVE_OUTPUT = Path('/content/drive/MyDrive/FYP-model-runs/qwen35-08b-lora-v3')
adapter_file = DRIVE_OUTPUT / 'best-adapter' / 'adapter_model.safetensors'
eval_file    = DRIVE_OUTPUT / 'evaluation.json'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(65536), b''):
            h.update(chunk)
    return h.hexdigest().upper()

metrics = json.loads(eval_file.read_text(encoding='utf-8')) if eval_file.is_file() else {}

summary = {
    'stage': 'v3',
    'completed_at': datetime.now(timezone.utc).isoformat(),
    'base_model': 'Qwen/Qwen3.5-0.8B',
    'adapter_sha256': sha256(adapter_file) if adapter_file.is_file() else None,
    'adapter_size_bytes': adapter_file.stat().st_size if adapter_file.is_file() else None,
    'final_metrics': metrics,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'unknown',
    'checkpoints': sorted([p.name for p in DRIVE_OUTPUT.glob('checkpoint-*') if p.is_dir()]),
    'hyperparameters': {
        'epochs': 3, 'learning_rate': 5e-5, 'lora_rank': 16,
        'lora_alpha': 32, 'effective_batch_size': 16,
        'max_length': 3072, 'early_stopping_patience': 1,
        'completion_only_loss': True,
    },
    'corpus': {
        'name': 'forecasting-llmrei-extraction-1049-v3',
        'train_records': 865, 'validation_records': 184,
        'task': 'extraction_only',
    }
}

summary_path = DRIVE_OUTPUT / 'v3-run-summary.json'
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'Run summary saved to: {summary_path}')
print(json.dumps(summary, indent=2))

---

## Troubleshooting

| Error | Fix |
|---|---|
| `ImportError: Found an incompatible version of torchao` | Re-run Cell 2 |
| `CUDA GPU not detected` | Runtime > Change runtime type > T4 GPU |
| `CalledProcessError` in Cell 3 (tests fail) | Already fixed — Cell 3 now sets PYTHONPATH and creates training/__init__.py |
| Colab disconnected mid-training | Re-run Cell 4 - auto-resume picks up the highest checkpoint |
| `best-adapter` missing after training | Check evaluation.json exists; if training exited early, re-run Cell 4 |
| Empty `slot_updates` in smoke test | v3 corpus is specifically designed to fix the v2 empty-update collapse |
| OOM (out of memory) | Restart runtime and re-run all cells from Cell 1 |

---

*Generated 2026-08-25 for the Local SLM / Forecasting-REI FYP.*